# inspecting datasets

In [46]:
from src.extract import extract, extract_all_datasets
import pandas as pd
import pathlib as pl

In [47]:
dataset_directory = pl.Path('datasets/brazilian-ecommerce/')
dataframes = extract_all_datasets(dataset_directory)

In [ ]:
for key, value in dataframes.items():
        # Investigating missing values

        # Getting datasets that has the null values
        # getting total column values for calculating total rows and columns
        column_with_nulls = value.columns[value.isna().any()].tolist()
        if column_with_nulls:
            print(key) # dataset name
            # Getting columns that has null valuesa
            for column_name in column_with_nulls:
                # Calculating percentage for null values
                null_counts = value[column_name].isna().sum()
                total_count = value.shape[0]
                null_value_percentage = (null_counts / total_count) * 100
                print(
                    f"{column_name}: {null_counts} missing values "
                    f"({null_value_percentage:.2f}%)"
                )

            print() # for new line

In [ ]:
 # Investigating orders dataset for order_delivered_customer_date, by checking orders status column and other things
orders_df = dataframes['olist_orders_dataset']

null_rows = orders_df[orders_df['order_delivered_customer_date'].isna()]

filtered_null_rows = null_rows.loc[
    :, 
        [
            
            'order_id', 
            'customer_id', 
            'order_status', 
            'order_approved_at',
            'order_purchase_timestamp', 
            'order_delivered_carrier_date', 'order_delivered_customer_date'
            
        ]
    ]

In [ ]:
# print(filtered_null_rows['order_status'].value_counts()) # check the order status wise counts

print(filtered_null_rows[filtered_null_rows['order_status'] == 'delivered'])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_delivered_carrier_date'].isna())])

In [ ]:
print(filtered_null_rows[(filtered_null_rows['order_status'] == 'delivered') & (filtered_null_rows['order_approved_at'].isna())])

# Task 06 - Investigate Product Information

In [ ]:
product_df = dataframes['olist_products_dataset']

In [ ]:
product_df

In [ ]:
filtered_products_df = product_df.loc[:, [
    'product_category_name', 
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty'
]]

In [ ]:
display(filtered_products_df.head())

In [ ]:
missing_product_category_name = filtered_products_df[filtered_products_df['product_category_name'].isna()]

In [ ]:
missing_product_category_name.isna().sum().any()

In [ ]:
missing_products = filtered_products_df[filtered_products_df['product_category_name'].isna() & filtered_products_df['product_name_lenght'].isna() & filtered_products_df['product_description_lenght'].isna() & filtered_products_df['product_photos_qty'].isna()]

In [ ]:
missing_products

In [ ]:
product_df[product_df['product_id'].isna()]

In [ ]:
missing_dimensions = product_df[
    product_df[
        [
            'product_weight_g',
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ].isna().any(axis=1)
]

In [ ]:
missing_dimensions

In [ ]:
reviews_df = dataframes['olist_order_reviews_dataset']

In [ ]:
reviews_df.columns.tolist()

In [ ]:
reviews_df.shape

In [ ]:
# checking duplicates
reviews_df[reviews_df.duplicated()]

In [ ]:
# Checking missing records
reviews_df['review_score'].isna().sum()

In [ ]:
# checking unique 
reviews_df['review_score'].unique()

In [ ]:
# Scorewise occurances
reviews_df['review_score'].value_counts()

In [ ]:
reviews_df['review_id'].duplicated().sum()

In [ ]:
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Missing identifiers
reviews_df[['review_id', 'order_id']].isna().sum()

In [ ]:
# Duplicate identifiers
reviews_df[['review_id', 'order_id']].duplicated().sum()

In [ ]:
# Check review_id uniqueness
reviews_df['review_id'].nunique()

In [ ]:
# Review score distribution
reviews_df['review_score'].value_counts().sort_index()

In [ ]:
duplicate_reviews = reviews_df[
    reviews_df['review_id'].duplicated(keep=False)
]

duplicate_reviews.sort_values('review_id')

In [ ]:
duplicate_reviews['review_id'].value_counts().head(30)

In [ ]:
sample_review_id = duplicate_reviews['review_id'].value_counts().index[0]

duplicate_reviews[
    duplicate_reviews['review_id'] == sample_review_id
].T

In [ ]:
duplicate_reviews.groupby('review_id')['order_id'].nunique().value_counts()

In [ ]:
duplicate_reviews.groupby('review_id')['review_score'].nunique().value_counts()

# Transformation starts

- [x] Identify timestamp columns.
- [x] Understand their current Pandas data types.
- [x] Decide which columns need conversion.
- [x] Implement the conversion.
- [x] Verify the result.

In [48]:
orders_df = dataframes['olist_orders_dataset']

In [ ]:
orders_df.dtypes

In [ ]:
orders_df.head()

In [ ]:
orders_df['order_status'].unique().tolist()

In [ ]:
orders_df['order_status'] = orders_df['order_status'].astype('category')

In [ ]:
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_approved_at'] = pd.to_datetime(orders_df['order_approved_at'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_delivered_carrier_date'] = pd.to_datetime(orders_df['order_delivered_carrier_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'], format="%Y-%m-%d %H:%M:%S", errors='coerce')

In [ ]:
orders_df[orders_df['order_purchase_timestamp'] == 'NaT']

# Task 08 - Investigate order timestamp consistency

### Part 1 — Investigate chronological consistency
- [x] Approval timestamp is earlier than purchase timestamp.
- [x] Carrier delivery timestamp is earlier than approval timestamp.
- [x] Customer delivery timestamp is earlier than carrier delivery timestamp.
- [x] Customer delivery timestamp is earlier than purchase timestamp.

In [ ]:
orders_df[orders_df['order_approved_at'] < orders_df['order_purchase_timestamp']].shape

In [ ]:
carrier_date_issued_data = orders_df[orders_df['order_delivered_carrier_date'] < orders_df['order_approved_at']]

In [ ]:
customer_date_issued_data = orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_delivered_carrier_date']]

In [ ]:
orders_df[orders_df['order_delivered_customer_date'] < orders_df['order_purchase_timestamp']].shape

In [ ]:
orders_df.head()

### Part 2 — Investigate estimated delivery
- [x] Estimated delivery date is earlier than purchase date.
- [x] Actual customer delivery is later than the estimated delivery date.

In [ ]:
orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_purchase_timestamp']].shape

In [ ]:
estimated_delivery_date_issued_data = orders_df[orders_df['order_estimated_delivery_date'] < orders_df['order_delivered_customer_date']]

### Part 3 — Report your findings
- [x] Number of affected records.
- [x] A few sample records (order_id and relevant timestamps).

In [ ]:
orders_df[['order_id', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']]

In [ ]:
carrier_date_issued_data.head(10)

In [ ]:
customer_date_issued_data.head(10)

In [ ]:
estimated_delivery_date_issued_data.shape

In [ ]:
estimated_delivery_date_issued_data.head(10)

In [ ]:
carrier_date_issued_data['order_status'].value_counts()

In [ ]:
customer_date_issued_data['order_status'].value_counts()

In [ ]:
estimated_delivery_date_issued_data['order_status'].value_counts()

In [ ]:
total_delivered_orders_count = orders_df['order_status'].value_counts().tolist()[0]

In [ ]:
late_delivered_orders = estimated_delivery_date_issued_data['order_status'].value_counts().tolist()[0]

In [ ]:
late_delivery_percentage = (late_delivered_orders / total_delivered_orders_count) * 100

In [ ]:
round(late_delivery_percentage, 2)

## Import transform_order function

In [ ]:
from src.transform import transform_orders

In [ ]:
copied_order_df = transform_orders(orders_df)

In [ ]:
copied_order_df.dtypes

In [ ]:
orders_df.dtypes

In [ ]:
copied_order_df.shape

In [ ]:
orders_df.shape

In [ ]:
timestamp_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
    ]

In [ ]:
for column in timestamp_columns:
    print(
        column,
        orders_df[column].isna().sum(),
        copied_order_df[column].isna().sum()
    )

In [ ]:
dataframes.keys()

In [ ]:
for key, value in dataframes.items():
    print(f'Dataset: {key}')
    for indx, column_name in enumerate(value.columns.tolist(), start=1):
        print(f"{indx} -> {column_name}")
    print()

# Task 13 A - Investigate order items

- [x] Data types of every column.
- [x] Missing values in every column.
- [x] Duplicate rows.
- [x] Whether order_id, order_item_id, and product_id contain missing values.
- [x] Whether price and freight_value contain zero or negative values.

In [ ]:
order_items_df = dataframes['olist_order_items_dataset']

In [ ]:
order_items_df.head()

In [ ]:
order_items_df.dtypes

In [ ]:
order_items_df.isna().sum()

In [ ]:
duplicates = order_items_df.duplicated(keep=False)
order_items_df[duplicates]

In [ ]:
zero_or_negative = order_items_df['price'] <= 0
order_items_df[zero_or_negative]

In [ ]:
order_items_df[order_items_df['freight_value'] <= 0]

In [ ]:
(order_items_df['freight_value'] == 0).sum()

In [ ]:
(order_items_df['freight_value'] < 0).sum()

In [ ]:
order_items_df[order_items_df['freight_value'] <= 0].head(20)

# Data Investigation on products dataset

In [ ]:
products_df = dataframes['olist_products_dataset']

In [ ]:
products_df.dtypes

In [ ]:
products_df.isna().sum()

In [ ]:
duplicates = products_df.duplicated(keep=False)
products_df[duplicates]

In [ ]:
query = (products_df['product_name_lenght'] <= 0) | (products_df['product_description_lenght'] <= 0) | (products_df['product_photos_qty'] <= 0) | (products_df['product_weight_g'] <= 0) | (products_df['product_length_cm'] <= 0) | (products_df['product_height_cm'] <= 0) | (products_df['product_width_cm'] <= 0)

In [ ]:
products_df[query].shape

In [ ]:
products_df[query]

In [ ]:
numeric_columns = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

products_df[numeric_columns].le(0).sum()

In [5]:
for key in dataframes.keys():
    print(key)

olist_sellers_dataset
product_category_name_translation
olist_orders_dataset
olist_order_items_dataset
olist_customers_dataset
olist_geolocation_dataset
olist_order_payments_dataset
olist_order_reviews_dataset
olist_products_dataset


In [6]:
sellers_df = dataframes['olist_sellers_dataset']

In [7]:
sellers_df.dtypes

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [8]:
sellers_df.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [9]:
duplicates = sellers_df.duplicated(keep=False)
sellers_df[duplicates]

,seller_id,seller_zip_code_prefix,seller_city,seller_state


In [10]:
product_category_df = dataframes['product_category_name_translation']

In [11]:
product_category_df.dtypes

product_category_name            str
product_category_name_english    str
dtype: object

In [12]:
product_category_df.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [14]:
duplicates = product_category_df.duplicated(keep=False)
product_category_df[duplicates]

,product_category_name,product_category_name_english


In [15]:
customers_df = dataframes['olist_customers_dataset']

In [16]:
customers_df.dtypes

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [17]:
customers_df.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [18]:
duplicates = customers_df.duplicated(keep=False)
customers_df[duplicates]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [19]:
geolocation_df = dataframes['olist_geolocation_dataset']

In [20]:
geolocation_df.dtypes

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

In [21]:
geolocation_df.isna().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [42]:
duplicates = geolocation_df.duplicated(subset=['geolocation_lat', 'geolocation_lng'], keep=False)
geolocation_df[duplicates].shape

(413907, 5)

In [24]:
geolocation_df.shape

(1000163, 5)

In [25]:
order_payments_df = dataframes['olist_order_payments_dataset']

In [26]:
order_payments_df.dtypes

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

In [27]:
order_payments_df.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [28]:
duplicates = order_payments_df.duplicated(keep=False)
order_payments_df[duplicates]

,order_id,payment_sequential,payment_type,payment_installments,payment_value


In [30]:
order_payments_df[order_payments_df['payment_value'] <= 0].shape

(9, 5)

In [32]:
order_payments_df[order_payments_df['payment_installments'] <= 0].shape

(2, 5)

In [33]:
reviews_df = dataframes['olist_order_reviews_dataset']

In [34]:
reviews_df.dtypes

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [35]:
reviews_df.shape

(99224, 7)

In [36]:
reviews_df.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [37]:
duplicates = reviews_df.duplicated(keep=False)
reviews_df[duplicates]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [38]:
reviews_df[reviews_df['review_score'] <= 0]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


# validating orders

In [49]:
orders_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [60]:
bool(orders_df['order_id'].isna().any())

False

In [61]:
null_customer_id = bool(orders_df['customer_id'].isna().any())

In [62]:
null_customer_id

False

In [74]:
orders_df['order_id'].duplicated(keep=False).any()

np.False_

In [79]:
# validate order statuses
statuses = [
        'delivered',
        'shipped',
        'canceled',
        'unavailable',
        'invoiced',
        'processing',
        'created',
        'approved'
    ]

bool(orders_df['order_status'].isin(statuses).all())

True